# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/owendewing/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/owendewing/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/owendewing/Owen-AI7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/owendewing/Owen-AI7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/owendewing/Owen-AI7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '0d31da'. Skipping!
Property 'summary' already exists in node 'd9c9bb'. Skipping!
Property 'summary' already exists in node '7e685a'. Skipping!
Property 'summary' already exists in node 'ac2ac1'. Skipping!
Property 'summary' already exists in node 'f8e375'. Skipping!
Property 'summary' already exists in node 'a9cecd'. Skipping!
Property 'summary' already exists in node '540adc'. Skipping!
Property 'summary' already exists in node 'cce6f2'. Skipping!
Property 'summary' already exists in node '75af78'. Skipping!
Property 'summary' already exists in node 'dbf061'. Skipping!
Property 'summary' already exists in node 'df081a'. Skipping!
Property 'summary' already exists in node 'b30c9a'. Skipping!
Property 'summary' already exists in node '6930d8'. Skipping!
Property 'summary' already exists in node '0c65b2'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '0d31da'. Skipping!
Property 'summary_embedding' already exists in node 'd9c9bb'. Skipping!
Property 'summary_embedding' already exists in node '540adc'. Skipping!
Property 'summary_embedding' already exists in node '7e685a'. Skipping!
Property 'summary_embedding' already exists in node 'df081a'. Skipping!
Property 'summary_embedding' already exists in node 'ac2ac1'. Skipping!
Property 'summary_embedding' already exists in node 'dbf061'. Skipping!
Property 'summary_embedding' already exists in node 'cce6f2'. Skipping!
Property 'summary_embedding' already exists in node 'f8e375'. Skipping!
Property 'summary_embedding' already exists in node 'a9cecd'. Skipping!
Property 'summary_embedding' already exists in node '75af78'. Skipping!
Property 'summary_embedding' already exists in node '0c65b2'. Skipping!
Property 'summary_embedding' already exists in node 'b30c9a'. Skipping!
Property 'summary_embedding' already exists in node '6930d8'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 480)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 480)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Whaat is the knoledge center and how does it h...,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not provide specific informat...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) specify regarding we...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) pertains to weeks of instructi...,single_hop_specifc_query_synthesizer
2,What is Volume 8?,[Inclusion of Clinical Work in a Standard Term...,"In the provided context, Volume 8 refers to Ch...",single_hop_specifc_query_synthesizer
3,What are Non-Term Characteristics in education...,[Non-Term Characteristics A program that measu...,Non-Term Characteristics refer to programs tha...,single_hop_specifc_query_synthesizer
4,Can you tell me about Volume 8 and how it affe...,[both the credit or clock hours and the weeks ...,Volume 8 discusses the effect of accelerated p...,single_hop_specifc_query_synthesizer
5,"How does accelerated progression, through addi...",[<1-hop>\n\nboth the credit or clock hours and...,"Accelerated progression, achieved by completin...",multi_hop_abstract_query_synthesizer
6,Different academic year definitions for progra...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that schools can define d...,multi_hop_abstract_query_synthesizer
7,disbursement for pell and direct loan how many...,[<1-hop>\n\nboth the credit or clock hours and...,both the credit or clock hours and the weeks o...,multi_hop_abstract_query_synthesizer
8,Volume 2 Volume 7 what is about academic year ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that Volume 2 covers acad...,multi_hop_specific_query_synthesizer
9,How does Volume 8 address the inclusion of cli...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,Volume 8 explains that clinical work conducted...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '84bf25'. Skipping!
Property 'summary' already exists in node 'bfa5c2'. Skipping!
Property 'summary' already exists in node '9be6f2'. Skipping!
Property 'summary' already exists in node 'bc89d9'. Skipping!
Property 'summary' already exists in node '7752dc'. Skipping!
Property 'summary' already exists in node 'ccb442'. Skipping!
Property 'summary' already exists in node '797bf4'. Skipping!
Property 'summary' already exists in node 'd1312c'. Skipping!
Property 'summary' already exists in node '9839c9'. Skipping!
Property 'summary' already exists in node '0d6cd1'. Skipping!
Property 'summary' already exists in node 'a8f47f'. Skipping!
Property 'summary' already exists in node 'b61c00'. Skipping!
Property 'summary' already exists in node 'de15e7'. Skipping!
Property 'summary' already exists in node 'da1b01'. Skipping!


...

Property 'summary_embedding' already exists in node '84bf25'. Skipping!
Property 'summary_embedding' already exists in node '9be6f2'. Skipping!
Property 'summary_embedding' already exists in node '7752dc'. Skipping!
Property 'summary_embedding' already exists in node 'bfa5c2'. Skipping!
Property 'summary_embedding' already exists in node 'de15e7'. Skipping!
Property 'summary_embedding' already exists in node 'bc89d9'. Skipping!
Property 'summary_embedding' already exists in node 'ccb442'. Skipping!
Property 'summary_embedding' already exists in node 'da1b01'. Skipping!
Property 'summary_embedding' already exists in node '797bf4'. Skipping!
Property 'summary_embedding' already exists in node '9839c9'. Skipping!
Property 'summary_embedding' already exists in node 'b61c00'. Skipping!
Property 'summary_embedding' already exists in node 'd1312c'. Skipping!
Property 'summary_embedding' already exists in node '0d6cd1'. Skipping!
Property 'summary_embedding' already exists in node 'a8f47f'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is Title IV?,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not explicitly define Title I...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(b)?,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) refers to the weeks of instruc...,single_hop_specifc_query_synthesizer
2,What does Chapter 3 specify regarding clinical...,[Inclusion of Clinical Work in a Standard Term...,Chapter 3 states that clinical work conducted ...,single_hop_specifc_query_synthesizer
3,What are Non-Term Characteristics in programs?,[Non-Term Characteristics A program that measu...,Non-Term Characteristics refer to programs tha...,single_hop_specifc_query_synthesizer
4,How does proration of direct loan eligibility ...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that if a student acceler...,multi_hop_abstract_query_synthesizer
5,"So, like, if clinical work is included in a st...",[<1-hop>\n\nInclusion of Clinical Work in a St...,Including clinical work in a standard term may...,multi_hop_abstract_query_synthesizer
6,How do nonstandard terms and payment periods r...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Nonstandard terms are defined as terms that do...,multi_hop_abstract_query_synthesizer
7,How do disbursement timing requirements differ...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour or non-term credit-hour programs...,multi_hop_abstract_query_synthesizer
8,Considering the detailed information in Volume...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 outlines that an academic year for Ti...,multi_hop_specific_query_synthesizer
9,Chapter 2 and Chapter 3 how do they relate to ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 explains that academic years must ha...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'Based on the provided context, the kinds of loans available include:\n\n- Direct Subsidized Loans\n- Direct Unsubsidized Loans\n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)\n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)\n- Federal SLS Loans (previously made under the FFEL Program)\n- Federal PLUS Loans (previously made under the FFEL Program)\n\nGraduate or professional students are eligible for Direct Unsubsidized Loans and Direct PLUS Loans, but not for Direct Subsidized Loans. Additionally, Direct PLUS Loans can be taken by parents on behalf of dependent students.\n\nHence, the main available loans are Direct Subsidized, Direct Unsubsidized, and Direct PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'sunny-dust-82' at:
https://smith.langchain.com/o/d432d5de-9a60-4d76-b22f-5ee5c2cfdd2f/datasets/8360f2e0-5d29-4821-b8bc-e97d00540377/compare?selectedSessions=16aa7bdf-df9d-4237-b9f4-2f472f4116f1




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does Volume 8 explain the impact of accele...,Volume 8 explains that exceptions to normal lo...,None,Volume 8 discusses that accelerated progressio...,1,0,0,6.314717,bff56644-484d-4322-a4d9-1b8ecde942d0,1f1fe20f-af04-4444-ad53-e38bd4d7bc95
1,whats appendix A and B about in disbursement t...,"Based on the provided context, Appendix B is r...",None,Appendix A provides guidance on calculating Pe...,0,0,0,3.931254,10814420-e43b-4f73-b66e-6cff3a591628,b914df5d-76e4-46fb-a3f8-9c7053ab9829
2,Chapter 2 and Chapter 3 how do they relate to ...,I don't know.,None,Chapter 2 explains that academic years must ha...,0,0,0,0.828765,be08edaf-5624-4098-9ff8-0cc327245969,4f44b8fc-aea8-4e09-a6d0-f08a340bc884
3,Considering the detailed information in Volume...,"Based on the provided context, Volumes 2 and 7...",None,Volume 2 outlines that an academic year for Ti...,1,1,0,7.670747,efd50f74-24fd-420e-bffb-2484eb4bf0de,25fa29c7-f751-4662-a8fa-bcfdc999b372
4,How do disbursement timing requirements differ...,Based on the provided context:\n\n- **Subscrip...,None,In clock-hour or non-term credit-hour programs...,1,1,0,6.834219,f2089d66-82c6-4bec-83cc-33f28673d944,c8afb1a0-4a11-4e53-8463-5ab323cb4ac9
5,How do nonstandard terms and payment periods r...,Nonstandard terms that are substantially equal...,None,Nonstandard terms are defined as terms that do...,1,1,0,6.614266,0d5e3e80-e304-4040-9dd8-0f3a5feef02e,823cf98e-b448-49a9-b59f-e653a1d297a6
6,"So, like, if clinical work is included in a st...","Based on the provided context, if clinical wor...",None,Including clinical work in a standard term may...,1,1,0,8.010246,0ffb511f-fef2-4a2f-8073-c70c4ad1824d,5c7e9a99-cab9-49ac-acdc-eceff2d9de7b
7,How does proration of direct loan eligibility ...,"Based on the provided context, proration of Di...",None,The context explains that if a student acceler...,1,0,0,5.431924,0663b5d5-a0a4-4da8-bef8-9ea1488752ae,c5f6680f-027c-48be-bbf9-6efa6673a01d
8,What are Non-Term Characteristics in programs?,Non-Term Characteristics in programs are as fo...,None,Non-Term Characteristics refer to programs tha...,1,1,0,3.598285,61426c85-1519-428a-b5e2-ad9a478bf996,fe814447-1132-4153-b93f-f463531feea1
9,What does Chapter 3 specify regarding clinical...,Chapter 3 specifies that periods of clinical w...,None,Chapter 3 states that clinical work conducted ...,1,1,0,3.691331,82ed08cf-8b5e-4ea3-b377-37ea35fb87c7,a97e1c29-2a39-4fea-8162-2e22e84774cd


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided, there are several types of loans available to students and their families:\n\n1. **Direct Subsidized Loans** – These loans are based on the student's financial need and the government pays the interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These loans are not based on financial need, and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These loans are available to parents of dependent students or to graduate/professional students themselves. They can cover the student’s Cost of Attendance (COA) minus other financial aid received, with no fixed loan limit, as long as eligibility requirements are met.\n\nAdditionally, if a parent of a dependent student cannot obtain a Direct PLUS Loan, the student may be eligible for additional Direct Unsubsidized Loan amounts.\n\nI hope this helps clarify the types of loans that might be available to support e

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'best-rabbit-88' at:
https://smith.langchain.com/o/d432d5de-9a60-4d76-b22f-5ee5c2cfdd2f/datasets/8360f2e0-5d29-4821-b8bc-e97d00540377/compare?selectedSessions=d01a3640-5634-45ab-a622-4e948407cd84




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does Volume 8 explain the impact of accele...,Thank you for your thoughtful question. Based ...,None,Volume 8 discusses that accelerated progressio...,1,0,1,5.477396,bff56644-484d-4322-a4d9-1b8ecde942d0,573e8f3e-ecef-4c7a-8a53-6dd3190b2ca9
1,whats appendix A and B about in disbursement t...,Thank you for your thoughtful question. From t...,None,Appendix A provides guidance on calculating Pe...,1,0,1,4.878789,10814420-e43b-4f73-b66e-6cff3a591628,394f564a-14ad-4cdf-bfa4-dbddcf59d813
2,Chapter 2 and Chapter 3 how do they relate to ...,Thank you for your thoughtful question. It’s c...,None,Chapter 2 explains that academic years must ha...,0,0,1,4.951740,be08edaf-5624-4098-9ff8-0cc327245969,d6dfc862-f95f-4b8e-8091-698dff8372fb
3,Considering the detailed information in Volume...,Thank you for your thoughtful question. It’s c...,None,Volume 2 outlines that an academic year for Ti...,1,0,1,6.890999,efd50f74-24fd-420e-bffb-2484eb4bf0de,474a900b-a362-4f2b-9540-bddcd0863730
4,How do disbursement timing requirements differ...,Thank you for your thoughtful question. From t...,None,In clock-hour or non-term credit-hour programs...,1,0,1,5.212913,f2089d66-82c6-4bec-83cc-33f28673d944,33ff80a2-9151-4605-bfac-9b6116d1d6ed
5,How do nonstandard terms and payment periods r...,Thank you for your thoughtful question. It sho...,None,Nonstandard terms are defined as terms that do...,1,1,1,5.736244,0d5e3e80-e304-4040-9dd8-0f3a5feef02e,ebb25bb0-43ca-47c1-a4ea-ed5028801afb
6,"So, like, if clinical work is included in a st...",Thank you for your thoughtful question! Based ...,None,Including clinical work in a standard term may...,1,0,1,4.598850,0ffb511f-fef2-4a2f-8073-c70c4ad1824d,7d2c4189-0556-4f41-8510-5ef2cbd2c853
7,How does proration of direct loan eligibility ...,Thank you for your thoughtful question. Based ...,None,The context explains that if a student acceler...,1,1,1,6.226694,0663b5d5-a0a4-4da8-bef8-9ea1488752ae,6b37e4b2-8b43-42b2-b668-e9d0be4c4e74
8,What are Non-Term Characteristics in programs?,Thank you for your thoughtful question. Based ...,None,Non-Term Characteristics refer to programs tha...,1,1,1,3.685343,61426c85-1519-428a-b5e2-ad9a478bf996,d44e9a7b-fca1-4fbd-85c3-5dafaf50e696
9,What does Chapter 3 specify regarding clinical...,Thank you for your thoughtful question. Based ...,None,Chapter 3 states that clinical work conducted ...,1,1,1,4.699455,82ed08cf-8b5e-4ea3-b377-37ea35fb87c7,7711d207-3c76-4afa-83b0-ef43e9c31e4b


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.